In [2]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")  
import matplotlib.pyplot as plt

import pandas as pd
import seaborn as sns
sys.path.insert(0, os.path.abspath("../.."))

from AML_BRATS.data.data_loading import get_dataset_folds, BRATSDataset
from AML_BRATS.models.train_unet import UNet
from AML_BRATS.models.train_FNC2 import SegNet
from AML_BRATS.models.metrics import calculate_dice, calculate_precision, calculate_recall

folds, _ = get_dataset_folds(
    "../../data/BraTS2020_training_data/content/data/meta_data.csv"
)

device = torch.device("cpu")


In [6]:
fold1_val = folds[0][1]
inputs = fold1_val[:]
ds = BRATSDataset(inputs, base_path=Path("../.."))

#model = SegNet(num_classes=3, dropout_p= 0.2, MC_Dropout=True, num_passes=20)
model = UNet(num_classes=3, batch_norm=True)
MODEL_NAME = (
    "UNET_HYD_25EPOCHS_adam_BNORM_LR0.0001_WD0.01_bce1_NOAUG_BS64_FOLD1_final_pruned_random_0.2"
)
#model = UNet(3, True)
#MODEL_NAME = (
#     "UNET_HYD_25EPOCHS_adam_BNORM_LR0.0001_WD0.01_bce1_NOAUG_BS64_FOLD1"
# )  # PRETTY GOOD
#MODEL_NAME = (
#    "UNET_HYD_12EPOCHS_adam_BNORM_LR0.0001_WD0.01_bce1_BS64_FOLD1"
#)  # BETTER?
model.load_state_dict(
    torch.load(f"../../model_pruned/{MODEL_NAME}.pkl", weights_only=True, map_location=device),
)
model.eval()

sample_indices = [2, 7, 15, 23, 31, 42, 50, 61, 74, 88]
sample_indices = list(range(75, 1000, 155))
# Threshold value for binary predictions
THRESHOLD = 0.9
fig, axes = plt.subplots(
    len(sample_indices), 3, figsize=(15, 4 * len(sample_indices))
)
for row, sample_index in enumerate(sample_indices):
    data = ds[sample_index]
    im = torch.from_numpy(data["image"]).unsqueeze(0)
    mask = np.moveaxis(data["mask"], 0, -1)
    with torch.no_grad():
        output = model(im)
    rgb = output[0].softmax(dim=0).permute(1, 2, 0).cpu().numpy()
    
    # Apply thresholding per channel to get binary RGB
    thresholded_rgb = (rgb > THRESHOLD).astype(float)
    
    # Compute Dice score between thresholded prediction and ground truth
    thresholded_tensor = torch.from_numpy(thresholded_rgb).unsqueeze(0)  # B=1, H, W, C
    thresholded_tensor = thresholded_tensor.permute(0, 3, 1, 2)  # B, C, H, W
    mask_tensor = torch.from_numpy(mask).unsqueeze(0).permute(0, 3, 1, 2).float()  # B, C, H, W
    dice_score = float(calculate_dice(thresholded_tensor, mask_tensor).item())

    # Compute precision and recall using metrics functions (handles masking and smoothing)
    precision_mean = float(calculate_precision(thresholded_tensor, mask_tensor).item())
    recall_mean = float(calculate_recall(thresholded_tensor, mask_tensor).item())
    # precision_mean = 0
    # recall_mean = 0

    # Display softmax prediction
    axes[row, 0].imshow(rgb[:, :, :])
    axes[row, 0].set_title(f"Softmax Pred - sample {sample_index}")
    
    # Display thresholded prediction as binary RGB
    axes[row, 1].imshow(thresholded_rgb)
    axes[row, 1].set_title(f"Thresholded (T={THRESHOLD}) - sample {sample_index}\nDice: {dice_score:.4f} | Prec: {precision_mean:.4f} | Rec: {recall_mean:.4f}")

    # Display ground truth mask
    axes[row, 2].imshow(mask)
    axes[row, 2].set_title(f"Ground Truth - sample {sample_index}")

plt.tight_layout()
plt.savefig(f"{MODEL_NAME}_plot.png", dpi=300, bbox_inches="tight")
plt.show()

/tmp/ipykernel_3896/1028918995.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
model = SegNet(num_classes=3, dropout_p=0.2, MC_Dropout=True, num_passes=20)
model.load_state_dict(torch.load(f"../../models/{MODEL_NAME}_final.pkl", map_location=device))
model.eval()

sample_indices = [2, 7, 15, 23, 31, 42, 50, 61, 74, 88]
sample_indices = list(range(75, 1000, 155))
# Threshold value for binary predictions
THRESHOLD = 0.9
fig, axes = plt.subplots(
    len(sample_indices), 4, figsize = (20, 4 * len(sample_indices))
)
num_classes = 3
class_uncertainties = {
    "Tumor1": [], # i dont know which color is which tumor
    "Tumor2": [],
    "Tumor3": [],
}

for row, sample_index in enumerate(sample_indices):
    data = ds[sample_index]
    im = torch.from_numpy(data["image"]).unsqueeze(0).to(device)
    mask = np.moveaxis(np.asarray(data["mask"]), 0, -1)

    with torch.no_grad():
        mean, variance, uncertainty = model.predict_with_uncertainty(im)

    probs     = mean[0]                          
    var_map   = variance[0].cpu().numpy()        
    unc_map   = uncertainty[0].cpu().numpy()     
    pred_class = probs.argmax(dim=0).cpu().numpy()   
    ground_truth         = np.argmax(mask, axis=-1)           

    #Entropy from mean probabilities
    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=0).cpu().numpy()  # (H, W)

    #Per-class uncertainty
    for c, name in enumerate(class_uncertainties.keys()):
        pixel_mask = (ground_truth == c)                   
        vals = var_map[c][pixel_mask]            
        if len(vals) > 0:
            class_uncertainties[name].extend(vals.tolist())

    #Ground truth
    axes[row, 0].imshow(ground_truth, cmap="tab10", vmin=0, vmax=num_classes - 1)
    axes[row, 0].set_title("Ground truth")
    axes[row, 0].axis("off")

    #Prediction
    axes[row, 1].imshow(pred_class, cmap="tab10", vmin=0, vmax=num_classes - 1)
    axes[row, 1].set_title("Prediction")
    axes[row, 1].axis("off")

    #Prediction + uncertainty overlay (high uncertainty areas will be highlighted)
    axes[row, 2].imshow(pred_class, cmap="tab10", vmin=0, vmax=num_classes - 1)
    axes[row, 2].imshow(unc_map, cmap="hot", alpha=0.6)  
    axes[row, 2].set_title("Prediction + uncertainty")
    axes[row, 2].axis("off")

    #Predictive entropy, how certain the model is at each pixel (high entropy = more uncertainty)
    im_entropy = axes[row, 3].imshow(entropy, cmap="hot")
    axes[row, 3].set_title("Predictive entropy")
    axes[row, 3].axis("off")
    plt.colorbar(im_entropy, ax=axes[row, 3], fraction=0.046)

plt.tight_layout()
plt.savefig(f"{MODEL_NAME}_uncertainty_plot.png", dpi=300, bbox_inches="tight")
plt.show()

#Boxplot of per-class uncertainties
records = []
for cls, values in class_uncertainties.items():
    for v in values:
        records.append({"Class": cls, "Uncertainty": v})

df = pd.DataFrame(records)
plt.figure(figsize=(8, 6))
sns.violinplot(data=pd.DataFrame(df), y="Class", x="Uncertainty", split=True)
plt.title("Distribution of predictive variance by class")
plt.ylabel("Density")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"boxplot_{MODEL_NAME}.png", dpi=300, bbox_inches="tight")
plt.show()

#tumor1 sharp spike long thing tail means its mostly certain about its predictions
#tumor2 has a broader peak mean more uncertain predictions accross predictions 
#tumor2 flat peak and wide distrubution its very uncertain about the predictions. could be the smallest tumor or/and lesser data points 

/tmp/ipykernel_18640/283353232.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_18640/283353232.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
